# KD TRAINING

In [ ]:
#!/usr/bin/env python3
"""
QAD training script (aligned with your YOLO training config)
"""

import sys
from pathlib import Path
import albumentations as A

# Add ultralytics source to path if needed
ULTRALYTICS_SRC = Path.cwd() / "src" / "ultralytics"
if ULTRALYTICS_SRC.exists() and str(ULTRALYTICS_SRC) not in sys.path:
    sys.path.insert(0, str(ULTRALYTICS_SRC))

from ultralytics import YOLO
from ultralytics.models.yolo.detect.kd_train import KD_Trainer

custom_transforms = [
    A.ToGray(p=0.03),
    A.GaussianBlur(
        blur_limit=[3, 3],
        p=0.1,
    ),
    A.MotionBlur(
        blur_limit=[3, 3],
        allow_shifted=False,
        angle_range=[0, 0],
        direction_range=[0, 0],
        p=0.1,
    ),
    A.Downscale(
        scale_range=[0.95, 1.0],
        # interpolation_pair={"upscale":0,"downscale":0}
        p=0.02,
    ),
    A.CLAHE(p=0.02),
    A.OneOf(
        [
            A.GaussNoise(
                std_range=[0.03, 0.05],
                mean_range=[-0.05, 0.05],
                noise_scale_factor=0.5,
                # p=0.01,
            ),
            A.ISONoise(
                # p=0.01
                ),
        ],
        p=0.02,
    ),
    A.ImageCompression(quality_range=(75, 95), p=0.8),
]

def main():
    teacher_model = YOLO(
        r"D:\phunp\OTS\rider_parts_detection\trained_models\rider_parts_det_dataset_20260420\20260421_2323_rider_parts_det_26m_p2_192_pretrain\weights\best.pt"
    ).model  # <-- lấy nn.Module
    model = YOLO(r"D:\phunp\OTS\rider_parts_detection\src\ultralytics-8.4.33-cus\ultralytics\cfg\models\26\yolo26m-p2.yaml").load(r"D:\phunp\OTS\rider_parts_detection\pretrained_models\20260421_1452_yolo26m_Pretrain\exported_models\exported_last.pt").model
    config = {
        # ================= MODEL =================
        "model=model,
        # "teacher=r"D:\phunp\OTS\rider_parts_detection\trained_models\rider_parts_det_dataset_20260420\20260414_1230_rider_parts_det_26s_160_v3\weights\best.pt",
        "teacher=teacher_model,
        "data=r"D:\phunp\OTS\rider_parts_detection\dataset\rider_parts_det_dataset_20260422_distill\data.yaml",

        # ================= TRAIN =================
        "epochs=300,
        "batch=64,
        "imgsz=192,
        "device=0,
        # "workers=0,
        "exist_ok=True,
        # "fraction=0.05,

        # giống config YOLO của bạn
        "optimizer="auto",
        "lr0=0.003,
        "cos_lr=True,
        "patience=35,

        # ================= AUGMENT =================
        "hsv_h=0.015,
        "hsv_s=0.5,
        "hsv_v=0.25,

        "scale=0.1,
        "translate=0.1,
        "degrees=5,
        "mosaic=0.0,
        "mixup=0.0,
        "shear=0,
        "perspective=0.0,
        "fliplr=0.5,
        "augmentations":custom_transforms,

        # KD settings (gần với distillation_loss="cwd")
        "distillation_loss="cwd",
        "cos_d_loss=True,
        "distill_loss_weight=0.3,
        "s_layers":["2", "4", "6", "8", "13", "16", "19", "22", "25", "28"],
        "t_layers":["2", "4", "6", "8", "13", "16", "19", "22", "25", "28"],

        # ================= OUTPUT =================
        "project=r"D:\phunp\OTS\rider_parts_detection\trained_models",
        "name=r"rider_parts_det_dataset_20260422_distill\20260422_1200_rider_parts_det_26mp2_192_kd",

        "plots=True,
        "val=True,
        "save=True,
    }

    print("=" * 60)
    print("=" * 60)
    print(f"  Model: {config['model']}")
    print(f"  Teacher: {config['teacher']}")
    print(f"  LR: {config['lr0']} | Batch: {config['batch']}")
    print("=" * 60)

    trainer = KD_Trainer(overrides=config)
    trainer.train()


if __name__ == "__main__":
    main()

W0422 12:04:01.343000 13528 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Transferred 420/962 items from pretrained weights
  Model: DetectionModel(
  (model): Sequential(
    (0): Conv(
      (conv): Conv2d(3, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (1): Conv(
      (conv): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(128, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (2): C3k2(
      (cv1): Conv(
        (conv): Conv2d(128, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(128, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (cv2): Conv(
        (conv): Conv2d(192, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(256, eps=0.001, momentum=0.03, affine=True, trac

# GTA KD TRAINING

In [ ]:
#!/usr/bin/env python3
"""
QAD training script (aligned with your YOLO training config)
"""

import sys
from pathlib import Path

# Add ultralytics source to path if needed
ULTRALYTICS_SRC = Path.cwd() / "src" / "ultralytics"
if ULTRALYTICS_SRC.exists() and str(ULTRALYTICS_SRC) not in sys.path:
    sys.path.insert(0, str(ULTRALYTICS_SRC))

from ultralytics import YOLO
from ultralytics.models.yolo.detect.gta_kd_train import GTA_KD_Trainer

import albumentations as A

custom_transforms = [
    A.ToGray(p=0.03),
    A.GaussianBlur(
        blur_limit=[3, 3],
        p=0.1,
    ),
    A.MotionBlur(
        blur_limit=[3, 3],
        allow_shifted=False,
        angle_range=[0, 0],
        direction_range=[0, 0],
        p=0.1,
    ),
    A.Downscale(
        scale_range=[0.95, 1.0],
        # interpolation_pair={"upscale":0,"downscale":0}
        p=0.02,
    ),
    A.CLAHE(p=0.02),
    A.OneOf(
        [
            A.GaussNoise(
                std_range=[0.03, 0.05],
                mean_range=[-0.05, 0.05],
                noise_scale_factor=0.5,
                # p=0.01,
            ),
            A.ISONoise(
                # p=0.01
                ),
        ],
        p=0.02,
    ),
    A.ImageCompression(quality_range=(75, 95), p=0.8),
]
def main():
    teacher_model = YOLO(
        r"D:\phunp\OTS\rider_parts_detection\trained_models\rider_parts_det_dataset_20260420\20260421_2323_rider_parts_det_26m_p2_192_pretrain\weights\best.pt"
    ).model  # <-- lấy nn.Module
    model = YOLO(r"D:\phunp\OTS\rider_parts_detection\src\ultralytics-8.4.33-cus\ultralytics\cfg\models\26\yolo26m-p2.yaml").load(r"D:\phunp\OTS\rider_parts_detection\pretrained_models\20260421_1452_yolo26m_Pretrain\exported_models\exported_last.pt").model
    config = {
        # ================= MODEL =================
        "model=model,
        # "teacher=r"D:\phunp\OTS\rider_parts_detection\trained_models\rider_parts_det_dataset_20260420\20260414_1230_rider_parts_det_26s_160_v3\weights\best.pt",
        "teacher=teacher_model,
        "data=r"D:\phunp\OTS\rider_parts_detection\dataset\rider_parts_det_dataset_20260422_distill\data.yaml",

        # ================= TRAIN =================
        "epochs=300,
        "batch=64,
        "imgsz=192,
        "device=0,
        # "workers=0,
        "exist_ok=True,
        # "fraction=0.05,

        # giống config YOLO của bạn
        "optimizer="auto",
        "lr0=0.0025,
        "cos_lr=True,
        "patience=35,

        # ================= AUGMENT =================
        "hsv_h=0.015,
        "hsv_s=0.5,
        "hsv_v=0.25,

        "scale=0.1,
        "translate=0.1,
        "degrees=5,
        "mosaic=0.0,
        "mixup=0.0,
        "shear=0,
        "perspective=0.0,
        "fliplr=0.5,

        "augmentations":custom_transforms,

        # KD settings (gần với distillation_loss="cwd")
        "distillation_loss="cwd",
        "cos_d_loss=True,
        "distill_loss_weight=0.3,
        "s_layers":["2", "4", "6", "8", "13", "16", "19", "22", "25", "28"],
        "t_layers":["2", "4", "6", "8", "13", "16", "19", "22", "25", "28"],

        # ================= OUTPUT =================
        "project=r"D:\phunp\OTS\rider_parts_detection\trained_models",
        "name=r"rider_parts_det_dataset_20260422_distill\20260422_1348_rider_parts_det_26mp2_192_gta_kd",

        "plots=True,
        "val=True,
        "save=True,
    }

    print("=" * 60)
    print("=" * 60)
    print(f"  Model: {config['model']}")
    print(f"  Teacher: {config['teacher']}")
    print(f"  LR: {config['lr0']} | Batch: {config['batch']}")
    print("=" * 60)

    trainer = GTA_KD_Trainer(overrides=config)
    trainer.train()


if __name__ == "__main__":
    main()

W0422 13:48:35.530000 26768 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Transferred 420/962 items from pretrained weights
  Model: DetectionModel(
  (model): Sequential(
    (0): Conv(
      (conv): Conv2d(3, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (1): Conv(
      (conv): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(128, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
      (act): SiLU(inplace=True)
    )
    (2): C3k2(
      (cv1): Conv(
        (conv): Conv2d(128, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(128, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (cv2): Conv(
        (conv): Conv2d(192, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn): BatchNorm2d(256, eps=0.001, momentum=0.03, affine=True, trac

# GTA KD TRAINING CLS MAPPING

In [1]:
#!/usr/bin/env python3
"""
QAD training script (aligned with your YOLO training config)
"""

import sys
from pathlib import Path

# Add ultralytics source to path if needed
ULTRALYTICS_SRC = Path.cwd() / "src" / "ultralytics"
if ULTRALYTICS_SRC.exists() and str(ULTRALYTICS_SRC) not in sys.path:
    sys.path.insert(0, str(ULTRALYTICS_SRC))

from ultralytics import YOLO
from ultralytics.models.yolo.detect.gta_kd_train import GTA_KD_Trainer

import albumentations as A

custom_transforms = [
    A.ToGray(p=0.01),
    A.GaussianBlur(
        blur_limit=[3, 3],
        p=0.1,
    ),
    A.MotionBlur(
        blur_limit=[3, 3],
        allow_shifted=False,
        angle_range=[0, 0],
        direction_range=[0, 0],
        p=0.1,
    ),
    A.Downscale(
        scale_range=[0.95, 1.0],
        # interpolation_pair={"upscale":0,"downscale":0}
        p=0.02,
    ),
    A.CLAHE(p=0.02),
    A.OneOf(
        [
            A.GaussNoise(
                std_range=[0.03, 0.05],
                mean_range=[-0.05, 0.05],
                noise_scale_factor=0.5,
                # p=0.01,
            ),
            A.ISONoise(
                intensity=(0.1, 0.25),
                # p=0.01
                ),
        ],
        p=0.03
    ),
    A.ImageCompression(quality_range=(75, 95), p=0.8),
]
def main():
    # teacher_model = YOLO(
    #     r"D:\phunp\OTS\rider_parts_detection\trained_models\rider_parts_det_dataset_20260423_distill\20260427_2253_rider_parts_det_26mp2_192_gta_kd\weights\best.pt"
    # ).model  # <-- lấy nn.Module
    # # model = YOLO(r"D:\phunp\OTS\rider_parts_detection\src\ultralytics-8.4.33-cus-kd\ultralytics\cfg\models\26\yolo26m-p2.yaml").load(r"D:\phunp\OTS\rider_parts_detection\pretrained_models\20260421_1452_yolo26m_Pretrain\exported_models\exported_last.pt").model
    # model = YOLO(r"D:\phunp\OTS\rider_parts_detection\src\ultralytics-8.4.33-cus-kd\ultralytics\cfg\models\26\yolo26x-p234.yaml").model
    config = {
        # ================= MODEL =================
        "model":r"yolo26s.pt",
        # "teacher=r"D:\phunp\OTS\rider_parts_detection\trained_models\rider_parts_det_dataset_20260420\20260414_1230_rider_parts_det_26s_160_v3\weights\best.pt",
        "teacher":r"D:\phunp\OTS\rider_parts_detection\trained_models\rider_parts_det_dataset_20260505\20260510_1530_rider_parts_det_26x_192\weights\best.pt",
        "data":r"D:\phunp\OTS\rider_parts_detection\training_datasets\rider_parts_det_dataset_20260508_ft\data.yaml",

        # ================= TRAIN =================
        "epochs":300,
        "batch":16,
        "imgsz":192,
        "device":0,
        # "workers=0,
        # "exist_ok":True,
        # "fraction=0.05,

        # giống config YOLO của bạn
        "optimizer":"AdamW",
        "lr0":0.001,
        "cos_lr":True,
        "patience":40,
        # "rect":True,

        # ================= AUGMENT =================
        "hsv_h":0.015,
        "hsv_s":0.5,
        "hsv_v":0.25,

        "scale":0.05,
        "translate":0.03,
        "degrees":5,
        "mosaic":0.0,
        "mixup":0.0,
        "shear":0,
        "perspective":0.0,
        "fliplr":0.5,

        "augmentations":custom_transforms,

        # KD settings (gần với distillation_loss="cwd")
        "distillation_loss":"cwd",
        "cos_d_loss":True,
        "distill_loss_weight":0.3,
        # "s_layers":["2", "4", "6", "16", "19", "22", "25"],
        # "t_layers":["2", "4", "6", "11", "14", "17", "20"],
        "s_layers":["6", "8", "13", "16", "19", "22"],
        "t_layers":["6", "8", "13", "16", "19", "22"],
        # "teacher_pred_conf":0.01,

        # ================= OUTPUT =================
        "project":r"D:\phunp\OTS\rider_parts_detection\trained_models",
        "name":r"rider_parts_det_dataset_20260508_ft\20260510_1713_rider_parts_det_26s_192_gta_kd",

        "plots":True,
        "val":True,
        "save":True,
    }

    # print(":" * 60)
    # print(":" * 60)
    # print(f"  Model: {config['model']}")
    # print(f"  Teacher: {config['teacher']}")
    # print(f"  LR: {config['lr0']} | Batch: {config['batch']}")
    # print(":" * 60)

    trainer = GTA_KD_Trainer(overrides=config)
    trainer.train()


if __name__ == "__main__":
    main()

Ultralytics 8.4.33  Python-3.12.0 torch-2.11.0+cu126 CUDA:0 (NVIDIA GeForce RTX 3060, 12288MiB)
models\yolo\detect\gta_kd_train: agnostic_nms=False, amp=True, angle=1.0, area_thr=0.0, augment=False, augmentations=[ToGray(p=0.01, method='weighted_average', num_output_channels=3), GaussianBlur(p=0.1, blur_limit=(3, 3), sigma_limit=(0.5, 3.0)), MotionBlur(p=0.1, allow_shifted=False, angle_range=(0.0, 0.0), blur_limit=(3, 3), direction_range=(0.0, 0.0)), Downscale(p=0.02, interpolation_pair={'upscale': 0, 'downscale': 0}, scale_range=(0.95, 1.0)), CLAHE(p=0.02, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8)), OneOf([
  GaussNoise(p=0.5, mean_range=(-0.05, 0.05), noise_scale_factor=0.5, per_channel=True, std_range=(0.03, 0.05)),
  ISONoise(p=0.5, color_shift=(0.01, 0.05), intensity=(0.1, 0.25)),
], p=0.03), ImageCompression(p=0.8, compression_type='jpeg', quality_range=(75, 95))], auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, class_mapping=None, classes=None, cl

d:\phunp\OTS\rider_parts_detection\src\ultralytics-8.4.33-cus-kd\.venv\Lib\site-packages\torch\jit\_trace.py:1022: UserWarning: The input to trace is already a ScriptModule, tracing it is a no-op. Returning the object as is.
  traced_func = _trace_impl(


TensorBoard: model graph visualization added 
Image sizes 192 train, 192 val
Using 8 dataloader workers
Logging results to D:\phunp\OTS\rider_parts_detection\trained_models\rider_parts_det_dataset_20260508_ft\20260510_1713_rider_parts_det_26s_192_gta_kd
Starting training for 300 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss    kd_loss  Instances       Size
      1/300      1.32G      1.408       1.49    0.02312     0.3923         18        192: 100% ━━━━━━━━━━━━ 56/56 1.3s/it 1:110.8sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 2.8it/s 6.7s0.1s
                   all        585       2997      0.274      0.273      0.338      0.137

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss    kd_loss  Instances       Size
      2/300      1.32G      1.187     0.5521    0.01827     0.3267         22        192: 100% ━━━━━━━━━━━━ 56/56 1.7it/s 33.1s0.4ss
                 Class     Images  Instances

# FINE TUNE

load trọng số từ nhiều scale

In [1]:
import torch
from ultralytics import YOLO

scale = "s"
# 1. Khởi tạo mô hình của bạn
model_path = rf"ultralytics\cfg\models\26\yolo26{scale}-p2.yaml"
student = YOLO(model_path)
s_layers = list(student.model.model)
new_student_sd = student.model.state_dict()

# 2. Danh sách các scale để tìm kiếm (thứ tự ưu tiên)
# list_weights = [r"D:\phunp\OTS\rider_parts_detection\pretrained_models\20260421_1452_yolo26m_Pretrain\exported_models\exported_last.pt", "yolo26l.pt", "yolo26s.pt", "yolo26n.pt"]
list_weights = ["yolo26s.pt", "yolo26m.pt", "yolo26l.pt", "yolo26n.pt", ]

# Tập hợp các index của student đã được nạp trọng số thành công
matched_student_indices = set()

print(f"--- Bắt đầu quy trình nhặt trọng số thông minh từ đa scale ---")

for w_path in list_weights:
    if len(matched_student_indices) == len([l for l in s_layers if list(l.state_dict().items())]):
        break # Đã tìm đủ tất cả các lớp có than số
        
    print(f"\n>> Đang tìm kiếm trong file: {w_path}")
    try:
        teacher = YOLO(w_path)
        t_layers = list(teacher.model.model)
    except Exception as e:
        print(f"Bỏ qua {w_path} do lỗi: {e}")
        continue

    used_teacher_indices = set()
    found_in_this_scale = 0

    for i, s_layer in enumerate(s_layers):
        # Nếu lớp này đã được nạp trọng số từ file trước đó, bỏ qua
        if i in matched_student_indices:
            continue
            
        s_type = s_layer.__class__.__name__
        s_params = list(s_layer.state_dict().items())
        
        if not s_params: # Lớp không có than số
            continue
            
        # Tìm kiếm theo kiến trúc trong scale hiện tại
        # Ưu tiên tìm các lớp có vị trí tương đồng
        search_order = sorted(range(len(t_layers)), key=lambda x: abs(x - i))
        
        for j in search_order:
            if j in used_teacher_indices: continue
            
            t_layer = t_layers[j]
            t_type = t_layer.__class__.__name__
            t_params = list(t_layer.state_dict().items())
            
            # Kiểm tra khớp Type và khớp Shape hoàn toàn
            if s_type == t_type and len(s_params) == len(t_params):
                if all(ps.shape == pt.shape for (_, ps), (_, pt) in zip(s_params, t_params)):
                    # Khớp hoàn toàn! Copy trọng số vào state_dict tổng
                    for (name_s, _), (name_t, val_t) in zip(s_params, t_params):
                        full_key = f"model.{i}.{name_s}"
                        new_student_sd[full_key] = val_t
                    
                    used_teacher_indices.add(j)
                    matched_student_indices.add(i)
                    found_in_this_scale += 1
                    # print(f" - Lớp [{i}] ({s_type}) lấy từ {w_path} lớp [{j}]")
                    break
                    
    print(f" => Lấy thêm được {found_in_this_scale} lớp mới từ {w_path}")

# 3. Nạp trọng số cuối cùng vào mô hình
student.model.load_state_dict(new_student_sd)

# 4. In báo cáo tổng kết
print(f"\n" + "="*50)
total_param_layers = len([l for l in s_layers if list(l.state_dict().items())])
print(f"KẾT QUẢ: Đã nạp {len(matched_student_indices)} / {total_param_layers} cụm lớp có than số.")

missing_indices = [i for i, l in enumerate(s_layers) if list(l.state_dict().items()) and i not in matched_student_indices]
if missing_indices:
    print(f"Các lớp vẫn trống (bắt buộc phải train):")
    for idx in missing_indices:
        print(f" - Lớp [{idx}]: {s_layers[idx].__class__.__name__}")
else:
    print("Chúc mừng! Tất cả các lớp đều đã được nạp trọng số tương ứng.")
print("="*50)


--- Bắt đầu quy trình nhặt trọng số thông minh từ đa scale ---

>> Đang tìm kiếm trong file: yolo26s.pt
 => Lấy thêm được 17 lớp mới từ yolo26s.pt

>> Đang tìm kiếm trong file: yolo26m.pt
 => Lấy thêm được 0 lớp mới từ yolo26m.pt

>> Đang tìm kiếm trong file: yolo26l.pt
 => Lấy thêm được 0 lớp mới từ yolo26l.pt

>> Đang tìm kiếm trong file: yolo26n.pt
 => Lấy thêm được 3 lớp mới từ yolo26n.pt

KẾT QUẢ: Đã nạp 20 / 21 cụm lớp có than số.
Các lớp vẫn trống (bắt buộc phải train):
 - Lớp [29]: Detect


In [1]:
#!/usr/bin/env python3
"""
QAD training script (aligned with your YOLO training config)
"""
from ultralytics import settings
settings.update({"tensorboard": True})

from ultralytics.data.augment import CustomCopyPasteTransform_AllBox
import sys
from pathlib import Path

# Add ultralytics source to path if needed
ULTRALYTICS_SRC = Path.cwd() / "src" / "ultralytics"
if ULTRALYTICS_SRC.exists() and str(ULTRALYTICS_SRC) not in sys.path:
    sys.path.insert(0, str(ULTRALYTICS_SRC))

from ultralytics import YOLO
from ultralytics.models.yolo.detect.gta_kd_train import GTA_KD_Trainer

import albumentations as A

custom_transforms = [
    # CustomCopyPasteTransform_AllBox(
    #     number_of_cpy=1,
    #     hard_non_overlay_classes=[0],
    #     soft_non_overlay_classes=[1, 2, 3],
    #     bbox_format="yolo",
    #     always_apply=True,
    #     p=1.0,
    # ),
    # A.ElasticTransform(
    #     alpha=100,
    #     sigma=10,
    #     interpolation=1,
    #     approximate=False,
    #     same_dxdy=True,
    #     mask_interpolation=0,
    #     noise_distribution="gaussian",
    #     keypoint_remapping_method="mask",
    #     border_mode=3,
    #     fill=114,
    #     fill_mask=0,
    #     p=0.2,
    # ),
    A.ToGray(p=0.01),
    A.GaussianBlur(
        blur_limit=[3, 3],
        p=0.1,
    ),
    A.MotionBlur(
        blur_limit=[3, 3],
        allow_shifted=False,
        angle_range=[0, 0],
        direction_range=[0, 0],
        p=0.1,
    ),
    A.Downscale(
        scale_range=[0.95, 1.0],
        # interpolation_pair={"upscale":0,"downscale":0}
        p=0.02,
    ),
    A.CLAHE(p=0.02),
    A.OneOf(
        [
            A.GaussNoise(
                std_range=[0.03, 0.05],
                mean_range=[-0.05, 0.05],
                noise_scale_factor=0.5,
                # p=0.01,
            ),
            A.ISONoise(
                intensity=(0.1, 0.25),
                # p=0.01
                ),
        ],
        p=0.03
    ),
    A.ImageCompression(quality_range=(75, 95), p=0.8),
]
# model = student 
scale="s"
# model = YOLO(r"yolo26s-p2.yaml").load(r"D:\phunp\OTS\rider_parts_detection\pretrained_models\20260507_0826_yolo26s_p2_Pretrain\exported_models\exported_last.pt")
model = YOLO(r"yolo12s.pt")
imgsz = 192
model.train(
        data=r"D:\phunp\OTS\rider_parts_detection\training_datasets\rider_parts_det_dataset_20260505\data.yaml",
        # ================= TRAIN =================
        epochs=300,
        batch=16,
        imgsz=imgsz,
        device=0,
        
        # epochs=1,
        # workers=0,
        # exist_ok=True,
        # fraction=0.05,

        # giống config YOLO của bạn
        optimizer="AdamW",
        # weight_decay=0.001,
        lr0=0.001,
        cos_lr=True,
        patience=35,
        # rect=True,

        # ================= AUGMENT =================
        hsv_h=0.015,
        hsv_s=0.5,
        hsv_v=0.25,

        scale=0.03,
        translate=0.03,
        degrees=5,
        mosaic=0.0,
        mixup=0.0,
        shear=0,
        perspective=0.0,
        fliplr=0.5,

        augmentations=custom_transforms,

        # ================= OUTPUT =================
        project=r"D:\phunp\OTS\rider_parts_detection\trained_models",
        name=rf"rider_parts_det_dataset_20260505\20260511_1121_rider_parts_det_12{scale}_{imgsz}",
)

New https://pypi.org/project/ultralytics/8.4.48 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.33  Python-3.12.0 torch-2.11.0+cu126 CUDA:0 (NVIDIA GeForce RTX 3060, 12288MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, area_thr=0.0, augment=False, augmentations=[ToGray(p=0.01, method='weighted_average', num_output_channels=3), GaussianBlur(p=0.1, blur_limit=(3, 3), sigma_limit=(0.5, 3.0)), MotionBlur(p=0.1, allow_shifted=False, angle_range=(0.0, 0.0), blur_limit=(3, 3), direction_range=(0.0, 0.0)), Downscale(p=0.02, interpolation_pair={'upscale': 0, 'downscale': 0}, scale_range=(0.95, 1.0)), CLAHE(p=0.02, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8)), OneOf([
  GaussNoise(p=0.5, mean_range=(-0.05, 0.05), noise_scale_factor=0.5, per_channel=True, std_range=(0.03, 0.05)),
  ISONoise(p=0.5, color_shift=(0.01, 0.05), intensity=(0.1, 0.25)),
], p=0.03), ImageCompression(p=0.8, compression_type='jpeg', quality_range=(75, 95))], auto_augment=randaugment, batc

d:\phunp\OTS\rider_parts_detection\src\ultralytics-8.4.33-cus-kd\.venv\Lib\site-packages\torch\jit\_trace.py:1022: UserWarning: The input to trace is already a ScriptModule, tracing it is a no-op. Returning the object as is.
  traced_func = _trace_impl(


TensorBoard: model graph visualization added 
Image sizes 192 train, 192 val
Using 8 dataloader workers
Logging results to D:\phunp\OTS\rider_parts_detection\trained_models\rider_parts_det_dataset_20260505\20260511_1121_rider_parts_det_12s_192
Starting training for 300 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      1/300     0.639G      1.123     0.9078      1.246          4        192: 100% ━━━━━━━━━━━━ 123/123 2.8it/s 44.1s0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 1.5it/s 12.6s0.3s
                   all        587       3003        0.7       0.72      0.735      0.499

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      2/300     0.727G      0.984     0.5661      1.121          4        192: 100% ━━━━━━━━━━━━ 123/123 4.7it/s 26.0s0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% 

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000017984F0B440>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
       

In [ ]:
# from ultralytics import YOLO

# # Load model
# model = YOLO(r"D:\phunp\OTS\rider_parts_detection\trained_models\rider_parts_det_dataset_20260505\20260505_1353_rider_parts_det_26m_p234_192_rect\weights\best.pt")

# # Xuất ra file "sạch" (giảm dung lượng)
# model.save(r"D:\phunp\OTS\rider_parts_detection\trained_models\rider_parts_det_dataset_20260505\20260505_1353_rider_parts_det_26m_p234_192_rect\weights\best-cleaned.pt") 


In [7]:
import torch

ckpt_path = r"D:\phunp\OTS\rider_parts_detection\trained_models\rider_parts_det_dataset_20260505\20260505_1353_rider_parts_det_26m_p234_192_rect\weights\best.pt"
checkpoint = torch.load(ckpt_path, map_location='cpu', weights_only=False)

# 1. Làm sạch train_args
if 'train_args' in checkpoint:
    args = checkpoint['train_args']
    if isinstance(args, dict):
        # Các key gây nặng nhất thường là 'model' (do chứa object model) 
        # và 'augmentations' (do chứa object transform)
        keys_to_clear = ['model', 'augmentations', 'teacher', 'cfg']
        
        for k in keys_to_clear:
            if k in args:
                print(f"Cleaning key: {k} from train_args")
                # Thay thế bằng giá trị đơn giản (string hoặc None)
                if k == 'model':
                    args[k] = "yolo26-p234.yaml"
                elif k == 'augmentations':
                    args[k] = None # Đây là thủ phạm khiến file nặng 74MB
                else:
                    args[k] = None

# 2. Xóa các key thừa ở cấp cao nhất
for key in ['optimizer', 'ema', 'scaler']:
    if key in checkpoint:
        checkpoint[key] = None

# 3. Lưu lại
final_path = r"D:\phunp\OTS\rider_parts_detection\trained_models\rider_parts_det_dataset_20260505\20260505_1353_rider_parts_det_26m_p234_192_rect\weights\best_final_25mb.pt"
torch.save(checkpoint, final_path)

print(f"Done! Check size: {final_path}")


Cleaning key: model from train_args
Cleaning key: teacher from train_args
Cleaning key: cfg from train_args
Done! Check size: D:\phunp\OTS\rider_parts_detection\trained_models\rider_parts_det_dataset_20260505\20260505_1353_rider_parts_det_26m_p234_192_rect\weights\best_final_25mb.pt


# FINETUNE MORE

In [4]:
#!/usr/bin/env python3
"""
QAD training script (aligned with your YOLO training config)
"""
from ultralytics import settings
settings.update({"tensorboard": True})

from ultralytics.data.augment import CustomCopyPasteTransform_AllBox
import sys
from pathlib import Path

# Add ultralytics source to path if needed
ULTRALYTICS_SRC = Path.cwd() / "src" / "ultralytics"
if ULTRALYTICS_SRC.exists() and str(ULTRALYTICS_SRC) not in sys.path:
    sys.path.insert(0, str(ULTRALYTICS_SRC))

from ultralytics import YOLO
from ultralytics.models.yolo.detect.gta_kd_train import GTA_KD_Trainer

import albumentations as A

custom_transforms = [
    # CustomCopyPasteTransform_AllBox(
    #     number_of_cpy=1,
    #     hard_non_overlay_classes=[0],
    #     soft_non_overlay_classes=[1, 2, 3],
    #     bbox_format="yolo",
    #     always_apply=True,
    #     p=1.0,
    # ),
    # A.ElasticTransform(
    #     alpha=100,
    #     sigma=10,
    #     interpolation=1,
    #     approximate=False,
    #     same_dxdy=True,
    #     mask_interpolation=0,
    #     noise_distribution="gaussian",
    #     keypoint_remapping_method="mask",
    #     border_mode=3,
    #     fill=114,
    #     fill_mask=0,
    #     p=0.2,
    # ),
    # A.ToGray(p=0.01),
    # A.GaussianBlur(
    #     blur_limit=[3, 3],
    #     p=0.1,
    # ),
    # A.MotionBlur(
    #     blur_limit=[3, 3],
    #     allow_shifted=False,
    #     angle_range=[0, 0],
    #     direction_range=[0, 0],
    #     p=0.1,
    # ),
    # A.Downscale(
    #     scale_range=[0.95, 1.0],
    #     # interpolation_pair={"upscale":0,"downscale":0}
    #     p=0.02,
    # ),
    # A.CLAHE(p=0.02),
    # A.OneOf(
    #     [
    #         A.GaussNoise(
    #             std_range=[0.03, 0.05],
    #             mean_range=[-0.05, 0.05],
    #             noise_scale_factor=0.5,
    #             # p=0.01,
    #         ),
    #         A.ISONoise(
    #             intensity=(0.1, 0.25),
    #             # p=0.01
    #             ),
    #     ],
    #     p=0.03
    # ),
    A.ImageCompression(quality_range=(75, 95), p=0.8),
]
# model = student 
scale="s"
model = YOLO(r"D:\phunp\OTS\rider_parts_detection\trained_models\rider_parts_det_dataset_20260505\20260511_1121_rider_parts_det_12s_192\weights\best.pt")
imgsz = 192
model.train(
        data=r"D:\phunp\OTS\rider_parts_detection\training_datasets\rider_parts_det_dataset_20260508_ft\data.yaml",
        # teacher=None,
        # ================= TRAIN =================
        epochs=300,
        batch=8,
        imgsz=imgsz,
        device=0,
        
        # epochs=1,
        # workers=0,
        # exist_ok=True,
        # fraction=0.05,

        # giống config YOLO của bạn
        optimizer="AdamW",
        weight_decay=0.0015,
        lr0=0.00025,
        cos_lr=True,
        patience=50,
        # rect=True,

        # ================= AUGMENT =================
        hsv_h=0.015,
        hsv_s=0.5,
        hsv_v=0.25,

        scale=0.03,
        translate=0.03,
        degrees=5,
        mosaic=0.0,
        mixup=0.0,
        shear=0,
        perspective=0.0,
        fliplr=0.5,

        augmentations=custom_transforms,

        # ================= OUTPUT =================
        project=r"D:\phunp\OTS\rider_parts_detection\trained_models",
        name=rf"rider_parts_det_dataset_20260508_ft\20260511_1718_rider_parts_det_12{scale}_{imgsz}",
)

New https://pypi.org/project/ultralytics/8.4.48 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.33  Python-3.12.0 torch-2.11.0+cu126 CUDA:0 (NVIDIA GeForce RTX 3060, 12288MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, area_thr=0.0, augment=False, augmentations=[ImageCompression(p=0.8, compression_type='jpeg', quality_range=(75, 95))], auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, class_mapping=None, classes=None, close_mosaic=10, cls=0.5, compile=False, concatenate=0.0, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_d_loss=True, cos_lr=True, cutmix=0.0, data=D:\phunp\OTS\rider_parts_detection\training_datasets\rider_parts_det_dataset_20260508_ft\data.yaml, degrees=5, deterministic=True, device=0, dfl=1.5, distill_loss_weight=0.3, distillation_loss=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, f

d:\phunp\OTS\rider_parts_detection\src\ultralytics-8.4.33-cus-kd\.venv\Lib\site-packages\torch\jit\_trace.py:1022: UserWarning: The input to trace is already a ScriptModule, tracing it is a no-op. Returning the object as is.
  traced_func = _trace_impl(


TensorBoard: model graph visualization added 
Image sizes 192 train, 192 val
Using 8 dataloader workers
Logging results to D:\phunp\OTS\rider_parts_detection\trained_models\rider_parts_det_dataset_20260508_ft\20260511_1718_rider_parts_det_12s_192
Starting training for 300 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      1/300     0.486G     0.5982     0.3304     0.8621         14        192: 100% ━━━━━━━━━━━━ 111/111 8.1it/s 13.6s<0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 37/37 6.6it/s 5.6s<0.1s
                   all        585       2997      0.829      0.814      0.808      0.606

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      2/300     0.486G     0.5455     0.2976     0.8424         14        192: 100% ━━━━━━━━━━━━ 111/111 11.9it/s 9.3s0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 1

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000001D0D5855100>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
       

# CHECK LOAD MODEL

In [10]:
# model = YOLO(r"ultralytics\cfg\models\26\yolo26m-p234.yaml").load(r"D:\phunp\OTS\rider_parts_detection\pretrained_models\20260421_1452_yolo26m_Pretrain\exported_models\exported_last.pt")
model = YOLO(r"ultralytics\cfg\models\26\yolo26m-p234.yaml").load(r"yolo26l.pt")
# model = model.load(r"yolo26m.pt")

Transferred 194/708 items from pretrained weights


In [ ]:
import torch
from ultralytics import YOLO
from copy import deepcopy

# 1. Khởi tạo mô hình student (của bạn) và mô hình teacher (mẫu)
model_path = r"ultralytics\cfg\models\26\yolo26m-p234.yaml"
student = YOLO(model_path)
teacher = YOLO("yolo26m.pt") # Load cả model để lấy cấu trúc

def match_and_load_weights(student_model, teacher_model):
    s_layers = list(student_model.model.model) # Danh sách các lớp Sequential
    t_layers = list(teacher_model.model.model)
    
    matched_count = 0
    # Dictionary chứa state_dict mới cho student
    new_student_sd = student_model.model.state_dict()
    
    # Để tránh việc một lớp teacher bị dùng lại nhiều lần không mong muốn, 
    # ta có thể đánh dấu các lớp teacher đã sử dụng (tùy chọn)
    used_teacher_indices = set()

    print(f"--- Bắt đầu so khớp kiến trúc ---")
    
    for i, s_layer in enumerate(s_layers):
        s_type = s_layer.__class__.__name__
        s_params = list(s_layer.state_dict().items())
        
        if not s_params: # Lớp không có than số (ví dụ: Upsample, Concat)
            continue
            
        found_match = False
        # Tìm trong teacher lớp nào có cùng Type và cùng Shape than số
        # Ưu tiên tìm lớp có index gần với index hiện tại nhất
        search_order = sorted(range(len(t_layers)), key=lambda x: abs(x - i))
        
        for j in search_order:
            if j in used_teacher_indices: continue
            
            t_layer = t_layers[j]
            t_type = t_layer.__class__.__name__
            t_params = list(t_layer.state_dict().items())
            
            # Kiểm tra khớp Type
            if s_type == t_type:
                # Kiểm tra khớp số lượng và hình dạng các than số (weight, bias, ...)
                if len(s_params) == len(t_params):
                    if all(ps.shape == pt.shape for (_, ps), (_, pt) in zip(s_params, t_params)):
                        # Khớp hoàn toàn kiến trúc! Thực hiện copy trọng số
                        s_sd = s_layer.state_dict()
                        t_sd = t_layer.state_dict()
                        
                        # Cập nhật vào state_dict tổng của student
                        for (name_s, _), (name_t, val_t) in zip(s_params, t_params):
                            full_key = f"model.{i}.{name_s}"
                            new_student_sd[full_key] = val_t
                        
                        used_teacher_indices.add(j)
                        print(f"Layer Student [{i}] ({s_type}) <== Khớp ==> Layer Teacher [{j}] ({t_type})")
                        matched_count += 1
                        found_match = True
                        break
        
        if not found_match:
            print(f"Layer Student [{i}] ({s_type}) --> KHÔNG tìm thấy lớp tương đương phù hợp.")

    # Load toàn bộ state_dict đã map vào student
    student_model.model.load_state_dict(new_student_sd)
    return matched_count

# Thực hiện so khớp
total_matched = match_and_load_weights(student, teacher)

print(f"\nTổng cộng đã khớp và nạp thành công {total_matched} cụm lớp kiến trúc.")

# Lưu lại model đã nạp weights (nếu muốn)
# student.save("yolo26m_p234_initialized.pt")


In [28]:
import torch
from ultralytics.utils.torch_utils import intersect_dicts
from ultralytics import YOLO

# 1. Khởi tạo model của bạn
model_path = r"ultralytics\cfg\models\26\yolo26m-p234.yaml"
model = YOLO(model_path)
model_sd = model.model.state_dict()

# 2. Danh sách các file weights để tìm kiếm (thứ tự ưu tiên từ trái qua phải)
# Bạn cần đảm bảo các file này có sẵn trong thư mục
list_weights = ["yolo26m.pt", "yolo26l.pt", "yolo26s.pt", "yolo26n.pt"]

final_intersected = {}

def get_state_dict(path):
    try:
        ckpt = torch.load(path, weights_only=False)
        weights = ckpt.get('ema') or ckpt.get('model') or ckpt
        return weights.float().state_dict() if hasattr(weights, 'state_dict') else weights
    except Exception as e:
        print(f"Không thể load {path}: {e}")
        return None

# 3. Lặp qua từng file weights để "nhặt" những lớp còn thiếu
for w_path in list_weights:
    weights_sd = get_state_dict(w_path)
    if weights_sd is None: continue
    
    # Thực hiện remap cho từng file weights (vì cấu trúc backbone p234 của bạn bị lệch)
    remapped_sd = {}
    for k, v in weights_sd.items():
        if k.startswith('model.9.'):
            new_key = k.replace('model.9.', 'model.7.')
            remapped_sd[new_key] = v
        elif k.startswith('model.10.'):
            new_key = k.replace('model.10.', 'model.8.')
            remapped_sd[new_key] = v
        elif any(k.startswith(f'model.{i}.') for i in range(7)):
            remapped_sd[k] = v
        else:
            remapped_sd[k] = v
            
    # Tìm các lớp khớp với model hiện tại
    current_intersected = intersect_dicts(remapped_sd, model_sd)
    
    # Chỉ lấy những lớp mà các file weights trước đó chưa tìm thấy
    added_count = 0
    for k, v in current_intersected.items():
        if k not in final_intersected:
            final_intersected[k] = v
            added_count += 1
            
    if added_count > 0:
        print(f"Lấy thêm được {added_count} lớp từ: {w_path}")
    
    # Nếu đã tìm đủ tất cả các lớp thì dừng lại
    if len(final_intersected) == len(model_sd):
        break

# 4. Thực hiện load trọng số cuối cùng
model.model.load_state_dict(final_intersected, strict=False)

# 5. In kết quả tổng hợp
print(f"\n--- TỔNG CỘNG: Đã nạp thành công {len(final_intersected)} / {len(model_sd)} than số ---")

mismatched = [k for k in model_sd.keys() if k not in final_intersected]
if mismatched:
    mismatched_layers = sorted(set(['.'.join(k.split('.')[:2]) for k in mismatched]))
    print(f"Các cụm lớp vẫn còn thiếu (thường do lệch shape hoặc cấu trúc mới):")
    for layer in mismatched_layers:
        print(f" - {layer}")

print("\nQuá trình 'nhặt' weights từ các scale hoàn tất!")


Lấy thêm được 338 lớp từ: yolo26m.pt
Lấy thêm được 327 lớp từ: yolo26s.pt

--- TỔNG CỘNG: Đã nạp thành công 665 / 762 than số ---
Các cụm lớp vẫn còn thiếu (thường do lệch shape hoặc cấu trúc mới):
 - model.10
 - model.13
 - model.22

Quá trình 'nhặt' weights từ các scale hoàn tất!


In [27]:
import torch
from ultralytics.utils.torch_utils import intersect_dicts
from ultralytics import YOLO

# 1. Khởi tạo model từ file yaml của bạn
model_path = r"ultralytics\cfg\models\26\yolo26m-p234.yaml"
model = YOLO(model_path)
model_sd = model.model.state_dict()

# 2. Load trọng số từ file .pt gốc
weights_path = "yolo26m.pt"
ckpt = torch.load(weights_path, weights_only=False)
weights = ckpt.get('ema') or ckpt.get('model') or ckpt
weights_sd = weights.float().state_dict() if hasattr(weights, 'state_dict') else weights

# 3. Tạo dictionary remap để xử lý lệch layer
remapped_sd = {}
for k, v in weights_sd.items():
    # Remap lớp 9 (SPPF gốc) -> lớp 7 (SPPF của bạn)
    if k.startswith('model.9.'):
        new_key = k.replace('model.9.', 'model.7.')
        remapped_sd[new_key] = v
    # Remap lớp 10 (C2PSA gốc) -> lớp 8 (C2PSA của bạn)
    elif k.startswith('model.10.'):
        new_key = k.replace('model.10.', 'model.8.')
        remapped_sd[new_key] = v
    # Các lớp backbone từ 0-6 vẫn giữ nguyên
    elif any(k.startswith(f'model.{i}.') for i in range(7)):
        remapped_sd[k] = v
    # Mặc định giữ nguyên các key khác (để intersect_dicts tự lọc)
    else:
        remapped_sd[k] = v

# 4. Tìm các lớp trùng khớp sau khi đã remap
intersected = intersect_dicts(remapped_sd, model_sd)

# 5. Thực hiện load trọng số vào model
model.model.load_state_dict(intersected, strict=False)

# 6. In kết quả kiểm tra
print(f"\n--- Đã nạp thành công {len(intersected)} than số vào model ---")
# Liệt kê một số lớp tiêu biểu đã load (để tránh in quá dài)
loaded_layers = sorted(set(['.'.join(k.split('.')[:2]) for k in intersected.keys()]))
print("Các cụm lớp đã load trọng số thành công:", loaded_layers)

# 7. In các lớp KHÔNG load được
mismatched = [k for k in model_sd.keys() if k not in intersected]
if mismatched:
    mismatched_layers = sorted(set(['.'.join(k.split('.')[:2]) for k in mismatched]))
    print(f"\n--- {len(mismatched)} than số KHÔNG được nạp (giữ nguyên mặc định) ---")
    print("Các cụm lớp thiếu trọng số (thường là phần head):", mismatched_layers)

print("\nHoàn tất quá trình remap và nạp weights!")



--- Đã nạp thành công 338 than số vào model ---
Các cụm lớp đã load trọng số thành công: ['model.0', 'model.1', 'model.13', 'model.16', 'model.17', 'model.19', 'model.2', 'model.20', 'model.22', 'model.23', 'model.3', 'model.4', 'model.5', 'model.6', 'model.7', 'model.8']

--- 424 than số KHÔNG được nạp (giữ nguyên mặc định) ---
Các cụm lớp thiếu trọng số (thường là phần head): ['model.10', 'model.13', 'model.16', 'model.17', 'model.19', 'model.20', 'model.22', 'model.23']

Hoàn tất quá trình remap và nạp weights!


In [24]:
import torch
from ultralytics.utils.torch_utils import intersect_dicts
from ultralytics import YOLO

model = YOLO(r"ultralytics\cfg\models\26\yolo26m-p2.yaml")

# 1. Đường dẫn weights bạn muốn load
weights_path = "yolo26m.pt"

# 2. Load state_dict từ file weights
ckpt = torch.load(weights_path, weights_only=False)
# Lấy model từ checkpoint (thường nằm trong key 'model' hoặc 'ema')
weights = ckpt.get('ema') or ckpt.get('model') or ckpt
weights_sd = weights.float().state_dict() if hasattr(weights, 'state_dict') else weights

# 3. Lấy state_dict của model hiện tại (model bạn vừa khởi tạo từ YAML)
model_sd = model.model.state_dict()

# 4. Tìm các lớp trùng khớp (giống cách Ultralytics thực hiện load)
intersected = intersect_dicts(weights_sd, model_sd)

# 5. In kết quả
print(f"--- Đã tìm thấy {len(intersected)} lớp có thể nạp weights ---")
for key in intersected.keys():
    print(key)

# 6. Bạn cũng có thể in ra các lớp KHÔNG load được (do lệch shape hoặc thiếu)
mismatched = [k for k in model_sd.keys() if k not in intersected]
if mismatched:
    print(f"\n--- {len(mismatched)} lớp KHÔNG được nạp (giữ nguyên mặc định) ---")
    # for key in mismatched: print(key) # Bỏ comment nếu muốn xem chi tiết


--- Đã tìm thấy 420 lớp có thể nạp weights ---
model.0.conv.weight
model.0.bn.weight
model.0.bn.bias
model.0.bn.running_mean
model.0.bn.running_var
model.0.bn.num_batches_tracked
model.1.conv.weight
model.1.bn.weight
model.1.bn.bias
model.1.bn.running_mean
model.1.bn.running_var
model.1.bn.num_batches_tracked
model.2.cv1.conv.weight
model.2.cv1.bn.weight
model.2.cv1.bn.bias
model.2.cv1.bn.running_mean
model.2.cv1.bn.running_var
model.2.cv1.bn.num_batches_tracked
model.2.cv2.conv.weight
model.2.cv2.bn.weight
model.2.cv2.bn.bias
model.2.cv2.bn.running_mean
model.2.cv2.bn.running_var
model.2.cv2.bn.num_batches_tracked
model.2.m.0.cv1.conv.weight
model.2.m.0.cv1.bn.weight
model.2.m.0.cv1.bn.bias
model.2.m.0.cv1.bn.running_mean
model.2.m.0.cv1.bn.running_var
model.2.m.0.cv1.bn.num_batches_tracked
model.2.m.0.cv2.conv.weight
model.2.m.0.cv2.bn.weight
model.2.m.0.cv2.bn.bias
model.2.m.0.cv2.bn.running_mean
model.2.m.0.cv2.bn.running_var
model.2.m.0.cv2.bn.num_batches_tracked
model.2.m.0.cv3.c

# RESUME

In [ ]:
import albumentations as A
from ultralytics import YOLO

custom_transforms = [
    A.ToGray(p=0.01),
    A.GaussianBlur(
        blur_limit=[3, 7],
        p=0.03,
    ),
    A.MotionBlur(
        blur_limit=[3, 7],
        allow_shifted=False,
        angle_range=[0, 0],
        direction_range=[0, 0],
        p=0.03,
    ),
    A.Downscale(
        scale_range=[0.9, 1.0],
        # interpolation_pair={"upscale":0,"downscale":0}
        p=0.02,
    ),
    A.CLAHE(p=0.02),
    A.OneOf(
        [
            A.GaussNoise(
                std_range=[0.03, 0.05],
                mean_range=[-0.05, 0.05],
                noise_scale_factor=0.5,
                # p=0.01,
            ),
            A.ISONoise(
                # p=0.01
                ),
        ],
        p=0.02,
    ),
    A.ImageCompression(quality_range=(50, 75), p=0.8),
]

model = YOLO(r"D:\phunp\OTS\lp_brand_signal_det\trained_models\lp_brand_signal_det_dataset_20260413_crop\20260414_1200_lp_brand_signal_det_26s_320\weights\last.pt")
model.train(
    resume=True,
    data=r"D:\phunp\OTS\lp_brand_signal_det\training_datasets\lp_brand_signal_det_dataset_20260413_crop\data.yaml",
    # workers=0,
    # exist_ok=True,
    # epochs=1,
    epochs=300,
    batch=16,
    imgsz=320,
    device=0,
    plots=True,
    # fraction=0.1,

    optimizer="auto",
    patience=100,
    lr0=0.001,
    multi_scale=0.25,
    cos_lr=True,

    # AUGMENTATION PARAMS
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.25,
    # hsv_h=0.0,
    # hsv_s=0.0,
    # hsv_v=0.0,

    # close_mosaic=200,
    scale=0.1,
    translate=0.1,
    degrees=3,
    mosaic=0.0,
    mixup=0.0,
    cutmix=0.0,
    concatenate=1.0,
    shear=0,
    perspective=0.0,
    fliplr=0.0,
    
    # val=False,

    project=r"D:\phunp\OTS\lp_brand_signal_det\trained_models",
    
    name=r"lp_brand_signal_det_dataset_20260413_crop\20260414_1200_lp_brand_signal_det_26s_320",
    # name=r"temp",

    min_area=0.0,
    area_thr=0.0,
    
    augmentations=custom_transforms,

    save_training_imgs=False,
    save_val=False,
    save_dir_img=r"C:\Users\Admin\Desktop\get_item",
    save_dir_lbl=r"C:\Users\Admin\Desktop\get_item",
)


New https://pypi.org/project/ultralytics/8.4.39 available  Update with 'pip install -U ultralytics'
WARNING Custom Albumentations transforms were used in the original training run but are not being restored. To preserve custom augmentations when resuming, you need to pass the 'augmentations' parameter again to get expected results. Example: 
model.train(resume=True, augmentations=[ToGray(p=0.01, method='weighted_average', num_output_channels=3), GaussianBlur(p=0.02, blur_limit=(3, 3), sigma_limit=(0.5, 3.0)), MotionBlur(p=0.01, allow_shifted=False, angle_range=(0.0, 0.0), blur_limit=(3, 3), direction_range=(0.0, 0.0)), Downscale(p=0.02, interpolation_pair={'upscale': 0, 'downscale': 0}, scale_range=(0.9, 1.0)), CLAHE(p=0.02, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8)), ImageCompression(p=0.8, compression_type='jpeg', quality_range=(50, 75))])
Ultralytics 8.4.33  Python-3.12.0 torch-2.11.0+cu126 CUDA:0 (NVIDIA GeForce RTX 3060, 12288MiB)
engine\trainer: agnostic_nms=False, amp=True, a

# VAL

In [1]:
from ultralytics import YOLO

conf = 0.3
imgsz = 640

train_ds = "lp_brand_signal_det_dataset_20260413_crop"

model_name = "20260414_1200_lp_brand_signal_det_26s_320"
test_ds = "helmet_PHAM_VAN_BACH_20251106_part1_1_5"

#####

model_path = fr"D:\phunp\OTS\lp_brand_signal_det\trained_models\{train_ds}\{model_name}\weights\best.pt"
src_path = r"D:\phunp\OTS\lp_brand_signal_det\test_datasets\preprocessed"

project = r"D:\phunp\OTS\lp_brand_signal_det\test_results"
name = fr"{train_ds}\20260415_1543_model_{model_name}\val\{test_ds}_conf{conf}_{imgsz}"

model = YOLO(model_path)
model.model.names = {i: str(i) for i in model.model.names} 
model.val(
    data=src_path + "\\" + test_ds + "\\data.yaml",
    project=project,
    name=name,

    conf=conf,
    imgsz=imgsz,
    batch=8,

    visualize=True,
    save_conf=True,
    device=0,
)

Ultralytics 8.4.33  Python-3.12.0 torch-2.11.0+cu126 CUDA:0 (NVIDIA GeForce RTX 3060, 12288MiB)
YOLO26s summary (fused): 122 layers, 9,467,115 parameters, 0 gradients, 20.5 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 501.3129.1 MB/s, size: 140.5 KB)
val: Scanning D:\phunp\OTS\lp_brand_signal_det\test_datasets\preprocessed\helmet_PHAM_VAN_BACH_20251106_part1_1_5\val... 1119 images, 6 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1119/1119 646.4it/s 1.7s0.1s
val: New cache created: D:\phunp\OTS\lp_brand_signal_det\test_datasets\preprocessed\helmet_PHAM_VAN_BACH_20251106_part1_1_5\val.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 140/140 2.7it/s 52.4s0.5ss
                   all       1119       8856      0.809      0.705      0.779      0.586
                     0        950       1635      0.835      0.772      0.842      0.573
                     1        225        246      0.778      0.598      0.688      0.

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000001D5A2456C00>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
    

# PREDICT

In [2]:
from ultralytics import YOLO

conf = 0.5
imgsz = [192, 128]
imgsz = 192
rect = True
model_file = "rider_parts_det_26s_192x128_20260511_1054"
train_ds = "rider_parts_det_dataset_20260508_ft"

model_name = "20260511_1054_rider_parts_det_26s_192_rect"
test_ds = r"Qua_so_ng_qd\2"
# test_ds = r"Qua_so_ng_qd\3"
# test_ds = "20260428_pvb"
# test_ds = "helmet_limit"
# test_ds = "20260121"
# test_ds = "overload_violation"
# test_ds = "passenger_limit_violation"

#####
model_path = fr"D:\phunp\OTS\rider_parts_detection\trained_models\{train_ds}\{model_name}\weights\{model_file}.pt"
src_path = r"D:\phunp\OTS\rider_parts_detection\test_datasets"

project = r"D:\phunp\OTS\rider_parts_detection\test_results"
name = fr"{train_ds}\20260511_1122_model_{model_name}\predict\{test_ds}_conf{conf}_{imgsz}_{model_file}.pt{"_no_rect" if not rect else ""}"

model = YOLO(model_path)
model.model.names = {i: str(i) for i in model.model.names} 
model.predict(
    source=src_path + "\\" + test_ds,
    project=project,
    name=name,
    workers=0,

    conf=conf,
    imgsz=imgsz,
    batch=16,

    # visualize=True,
    save_preprocessed_img=False,
    save_img_dir=r"C:\Users\Admin\Desktop\preprocessed",
    save=True,
    save_txt=True,
    save_conf=True,
    device=0,
)


image 1/394 D:\phunp\OTS\rider_parts_detection\test_datasets\Qua_so_ng_qd\2\00_16_39_617__6438__42300___0.904785__ocr_35H302825_0.89447__origin_obj.jpeg: 192x192 2 0s, 1 1, 1 2, 12.8ms
image 2/394 D:\phunp\OTS\rider_parts_detection\test_datasets\Qua_so_ng_qd\2\00_33_13_037__3630__36896___0.905762__ocr_19AA70698_0.905762__origin_obj.jpeg: 192x192 2 0s, 1 1, 1 2, 1 3, 12.8ms
image 3/394 D:\phunp\OTS\rider_parts_detection\test_datasets\Qua_so_ng_qd\2\00_48_51_836__10270__61829___0.90332__ocr_15G193160_0.905701__origin_obj.jpeg: 192x192 2 0s, 1 1, 1 2, 1 3, 12.8ms
image 4/394 D:\phunp\OTS\rider_parts_detection\test_datasets\Qua_so_ng_qd\2\01_06_33_648__8256__66321___0.903809__ocr_28D10786_0.89624__origin_obj.jpeg: 192x192 2 0s, 1 1, 1 2, 12.8ms
image 5/394 D:\phunp\OTS\rider_parts_detection\test_datasets\Qua_so_ng_qd\2\02_44_54_103__3447__52622___0.899902__ocr_36A61792_0.905973__origin_obj.jpeg: 192x192 2 0s, 1 1, 1 2, 12.8ms
image 6/394 D:\phunp\OTS\rider_parts_detection\test_datasets\Qu

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: '0', 1: '1', 2: '2', 3: '3'}
 obb: None
 orig_img: array([[[119, 125, 108],
         [118, 124, 107],
         [117, 123, 106],
         ...,
         [ 97,  97,  97],
         [ 97,  97,  97],
         [ 96,  96,  96]],
 
        [[118, 124, 107],
         [118, 124, 107],
         [118, 124, 107],
         ...,
         [ 99,  99,  99],
         [ 98,  98,  98],
         [ 97,  97,  97]],
 
        [[118, 124, 107],
         [118, 124, 107],
         [117, 123, 106],
         ...,
         [ 99,  99,  99],
         [ 98,  98,  98],
         [ 98,  98,  98]],
 
        ...,
 
        [[ 95,  94,  98],
         [ 96,  95,  99],
         [ 98,  97, 101],
         ...,
         [ 74,  68,  79],
         [ 73,  67,  78],
         [ 74,  68,  79]],
 
        [[ 95,  94,  98],
         [ 96,  95,  99],
         [ 98,  97, 101],
         ...,

In [6]:
from ultralytics import YOLO

# Nạp model ONNX thay vì file .pt
model = YOLO(r"D:\phunp\OTS\rider_parts_detection\trained_models\rider_parts_det_dataset_20260423\20260504_1010_rider_parts_det_26m_p234_192\weights\best.pt", task="detect") 

model.model.names = {i: str(i) for i in model.model.names} 
# Chạy predict
results = model.predict(
    source=r"D:\phunp\OTS\rider_parts_detection\training_datasets\rider_parts_det_dataset_20260423\valid\qua_so_ng_qd_20260422", 
    conf=0.3,
    batch=32,
    imgsz=[192, 96], 
    save=True,
    save_txt=True,
    save_conf=True,
    show_conf=False,
    project=r"D:\phunp\OTS\rider_parts_detection\test_results\rider_parts_det_dataset_20260423\20260504_1335_model_20260504_1010_rider_parts_det_26m_p234_192\predict\Qua_so_ng_qd",
    name=r"val",
    )

# Duyệt kết quả
# for r in results:
#     print(r.boxes.data) # Tọa độ bounding box


image 1/208 D:\phunp\OTS\rider_parts_detection\training_datasets\rider_parts_det_dataset_20260423\valid\qua_so_ng_qd_20260422\00_16_39_617__6438__42300___0.904785__ocr_35H302825_0.89447__origin_obj.jpeg: 192x96 2 0s, 1 1, 1 2, 8.4ms
image 2/208 D:\phunp\OTS\rider_parts_detection\training_datasets\rider_parts_det_dataset_20260423\valid\qua_so_ng_qd_20260422\00_33_13_037__3630__36896___0.905762__ocr_19AA70698_0.905762__origin_obj.jpeg: 192x96 2 0s, 1 1, 1 2, 1 3, 8.4ms
image 3/208 D:\phunp\OTS\rider_parts_detection\training_datasets\rider_parts_det_dataset_20260423\valid\qua_so_ng_qd_20260422\00_48_51_836__10270__61829___0.90332__ocr_15G193160_0.905701__origin_obj.jpeg: 192x96 2 0s, 1 1, 1 2, 1 3, 8.4ms
image 4/208 D:\phunp\OTS\rider_parts_detection\training_datasets\rider_parts_det_dataset_20260423\valid\qua_so_ng_qd_20260422\01_06_33_648__8256__66321___0.903809__ocr_28D10786_0.89624__origin_obj.jpeg: 192x96 2 0s, 1 1, 1 2, 1 3, 8.4ms
image 5/208 D:\phunp\OTS\rider_parts_detection\trai

# OLD VERSION MODEL DETECT

In [ ]:
# %pip install --no-cache-dir "onnx" "onnxruntime-gpu"

In [ ]:
from ultralytics import YOLO

# Nạp model ONNX thay vì file .pt
model = YOLO(r"D:\phunp\OTS\lp_brand_signal_det\trained_models\lp_brand_signal_det_320_20260119.pt", task="detect") 

# Chạy predict
results = model.predict(
    source=r"D:\phunp\OTS\lp_brand_signal_det\test_datasets\helmet_PHAM_VAN_BACH_20251106\part1_col", 
    conf=0.25,
    batch=32,
    imgsz=320, 
    save=True,
    project=r"D:\phunp\OTS\lp_brand_signal_det\test_datasets\helmet_PHAM_VAN_BACH_20251106",
    name=r"part1_col_old_pred",
    )

# Duyệt kết quả
# for r in results:
#     print(r.boxes.data) # Tọa độ bounding box


Saving D:\phunp\OTS\lp_brand_signal_det\test_results\lp_brand_signal_det_dataset_20260413_crop\20260415_1543_model_20260414_1200_lp_brand_signal_det_26s_320\predict\helmet_PHAM_VAN_BACH_20251106_part1_1_5_conf0.3_320\concat_00000\stage0_Conv_features.png... (32/32)
Saving D:\phunp\OTS\lp_brand_signal_det\test_results\lp_brand_signal_det_dataset_20260413_crop\20260415_1543_model_20260414_1200_lp_brand_signal_det_26s_320\predict\helmet_PHAM_VAN_BACH_20251106_part1_1_5_conf0.3_320\concat_00000\stage1_Conv_features.png... (32/64)
Saving D:\phunp\OTS\lp_brand_signal_det\test_results\lp_brand_signal_det_dataset_20260413_crop\20260415_1543_model_20260414_1200_lp_brand_signal_det_26s_320\predict\helmet_PHAM_VAN_BACH_20251106_part1_1_5_conf0.3_320\concat_00000\stage2_C3k2_features.png... (32/128)
Saving D:\phunp\OTS\lp_brand_signal_det\test_results\lp_brand_signal_det_dataset_20260413_crop\20260415_1543_model_20260414_1200_lp_brand_signal_det_26s_320\predict\helmet_PHAM_VAN_BACH_20251106_part1

# PLOTS RESULT.PNG FROM .CSV FILE

In [3]:
from ultralytics.utils.plotting import plot_results
from pathlib import Path

# Đường dẫn tới file csv của bạn
csv_path = r"D:\phunp\OTS\rider_parts_detection\trained_models\rider_parts_det_dataset_20260505\20260511_1403_rider_parts_det_12s_192_rect\results.csv"

# Thực hiện vẽ
plot_results(file=csv_path)

print(f"Ảnh kết quả đã được lưu tại: {Path(csv_path).parent / 'results.png'}")


Ảnh kết quả đã được lưu tại: D:\phunp\OTS\rider_parts_detection\trained_models\rider_parts_det_dataset_20260505\20260511_1403_rider_parts_det_12s_192_rect\results.png


# STRIP OPTIMIZER

In [1]:
from ultralytics.utils.torch_utils import strip_optimizer
from pathlib import Path

# 1. Đường dẫn tới file weight bạn muốn làm gọn
weight_file = r"D:\phunp\OTS\rider_parts_detection\trained_models\rider_parts_det_dataset_20260505\20260505_1353_rider_parts_det_26m_p234_192_rect\weights\best.pt"
s=r"D:\phunp\OTS\rider_parts_detection\trained_models\rider_parts_det_dataset_20260505\20260505_1353_rider_parts_det_26m_p234_192_rect\weights\stripped.pt"
# 2. Gọi hàm strip_optimizer
# Nó sẽ tự động: chuyển sang FP16, xoá optimizer, xoá loss criterion và lưu đè lên file cũ
# strip_optimizer(weight_file)
strip_optimizer(weight_file, s=s)

print(f"Đã hoàn tất làm gọn model: {weight_file}")


Optimizer stripped from D:\phunp\OTS\rider_parts_detection\trained_models\rider_parts_det_dataset_20260505\20260505_1353_rider_parts_det_26m_p234_192_rect\weights\best.pt, saved as D:\phunp\OTS\rider_parts_detection\trained_models\rider_parts_det_dataset_20260505\20260505_1353_rider_parts_det_26m_p234_192_rect\weights\stripped.pt, 127.0MB
Đã hoàn tất làm gọn model: D:\phunp\OTS\rider_parts_detection\trained_models\rider_parts_det_dataset_20260505\20260505_1353_rider_parts_det_26m_p234_192_rect\weights\best.pt
